# XGBoost Regression – Untuned (GPU Compatible)

This notebook trains two **XGBoost regression models**:
- One using **all high-variance features** (`VarianceThreshold`)
- One using the **top 30 features** selected with `RandomForestRegressor`

The objective is to provide a strong baseline without any hyperparameter tuning, but with GPU acceleration.

In [8]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

try:
    booster = xgb.Booster(params={'tree_method':'gpu_hist'})
    print("GPU support detected: tree_method='gpu_hist' works.")
except Exception as e:
    print("GPU support NOT detected:", e)

try:
    booster = xgb.Booster(params={'tree_method':'hist', 'device':'cuda'})
    print("GPU support detected: device='cuda' works.")
except Exception as e:
    print("GPU support NOT detected:", e)


XGBoost version: 3.0.2
GPU support detected: tree_method='gpu_hist' works.
GPU support detected: device='cuda' works.


In [ ]:
import sys, os
# Add the project root to the Python path
project_root = os.path.abspath("../../..")
sys.path.append(project_root)

import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold
from utils.model_evaluator import ModelEvaluator

from utils.constants import ML_READY_DATA_FILE, TEST_MODE
from utils.data_loader import DataLoader
from utils.train_test_metrics_logger import TrainTestMetricsLogger
from utils.model_saver import ModelSaver

def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Display current test mode status
if TEST_MODE:
    print("TEST_MODE is ON – reduced data and iterations.")
else:
    print("TEST_MODE is OFF – full training.")

# === Load and prepare data ===
loader = DataLoader(ML_READY_DATA_FILE)
df = loader.load_data()
X = df.drop(columns=["price"])
y = df["price"]

# === Feature selection using variance threshold ===
selector = VarianceThreshold(threshold=0.01)
selector.fit(X)
X_reduced = X.loc[:, selector.get_support()]

# === Select top 30 features using Random Forest importance ===
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_reduced, y)
importances = rf.feature_importances_
top_features = pd.Series(importances, index=X_reduced.columns).sort_values(ascending=False).head(30).index.tolist()
X_top = X_reduced[top_features]

# === Split data for all features and top features ===
X_train, X_test, y_train, y_test = train_test_split(X_reduced, y, test_size=0.2, random_state=42)
X_train_top, X_test_top, _, _ = train_test_split(X_top, y, test_size=0.2, random_state=42)

# === Define XGBoost parameters ===
use_gpu = True
params = {
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_estimators": 100 if TEST_MODE else 400,
    "tree_method": "hist",
    "device": "cuda" if use_gpu else "cpu",
    "n_jobs": -1,
    "verbosity": 2
}

# === Train XGBoost on all features ===
model_all = xgb.XGBRegressor(**params)
model_all.fit(X_train, y_train)
y_pred_train_all = model_all.predict(X_train)
y_pred_test_all = model_all.predict(X_test)

# === Train XGBoost on top 30 features ===
model_top = xgb.XGBRegressor(**params)
model_top.fit(X_train_top, y_train)
y_pred_train_top = model_top.predict(X_train_top)
y_pred_test_top = model_top.predict(X_test_top)

# === Step 6: Evaluate models using ModelEvaluator ===

# Define price bins for evaluation by price range
price_bins = bins=[0, 250_000, 500_000, 750_000, 1_000_000, float("inf")]

# Evaluate – All Features
evaluator_all = ModelEvaluator("XGBoost CV (All Features)")
metrics_train_all, _ = evaluator_all.evaluate(y_train, y_pred_train_all, bins=price_bins, dataset_type="Train")
metrics_test_all, _ = evaluator_all.evaluate(y_test, y_pred_test_all, bins=price_bins, dataset_type="Test")
evaluator_all.print_evaluation(y_test, y_pred_test_all, bins=price_bins)

# Evaluate – Top 30 Features
evaluator_top = ModelEvaluator("XGBoost CV (Top RF Features)")
metrics_train_top, _ = evaluator_top.evaluate(y_train, y_pred_train_top, bins=price_bins, dataset_type="Train")
metrics_test_top, _ = evaluator_top.evaluate(y_test, y_pred_test_top, bins=price_bins, dataset_type="Test")
evaluator_top.print_evaluation(y_test, y_pred_test_top, bins=price_bins)

# === Step 7: Log results ===
logger = TrainTestMetricsLogger()

logger.log(
    model_name=f"XGBoost CV (All Features){' [TEST]' if TEST_MODE else ''}",
    experiment_name=f"XGBoost Untuned (All Features){' [TEST]' if TEST_MODE else ''}",
    mae_train=metrics_train_all["mae"],
    rmse_train=metrics_train_all["rmse"],
    r2_train=metrics_train_all["r2"],
    mae_test=metrics_test_all["mae"],
    rmse_test=metrics_test_all["rmse"],
    r2_test=metrics_test_all["r2"],
    data_file=ML_READY_DATA_FILE,
    n_features=X_train.shape[1]
)

logger.log(
    model_name=f"XGBoost CV (Top RF Features){' [TEST]' if TEST_MODE else ''}",
    experiment_name=f"XGBoost Untuned (Top RF Features){' [TEST]' if TEST_MODE else ''}",
    mae_train=metrics_train_top["mae"],
    rmse_train=metrics_train_top["rmse"],
    r2_train=metrics_train_top["r2"],
    mae_test=metrics_test_top["mae"],
    rmse_test=metrics_test_top["rmse"],
    r2_test=metrics_test_top["r2"],
    data_file=ML_READY_DATA_FILE,
    n_features=X_train_top.shape[1]
)

# === Step 8: Display summary ===
logger.display_table()


# Step 9: Save models, features and metrics
saver = ModelSaver()

saver.save_model_and_features(
    model=model_all,
    features=X_train.columns.tolist(),
    model_name=f"XGBoost CV (All Features){' [TEST]' if TEST_MODE else ''}",
    metrics=metrics_test_all,
    metrics_by_price_range=evaluator_all._compute_metrics_by_price_range(y_test, y_pred_test_all, bins=price_bins)
)

saver.save_model_and_features(
    model=model_top,
    features=X_train_top.columns.tolist(),
    model_name=f"XGBoost CV (Top RF Features){' [TEST]' if TEST_MODE else ''}",
    metrics=metrics_test_top,
    metrics_by_price_range=evaluator_top._compute_metrics_by_price_range(y_test, y_pred_test_top, bins=price_bins)
)


TEST_MODE is OFF – full training.
[11:26:53] INFO: C:\actions-runner\_work\xgboost\xgboost\src\data\iterative_dmatrix.cc:53: Finished constructing the `IterativeDMatrix`: (16111, 72, 1159992).
[11:26:53] INFO: C:\actions-runner\_work\xgboost\xgboost\src\data\ellpack_page.cu:167: Ellpack is dense.
[11:26:55] INFO: C:\actions-runner\_work\xgboost\xgboost\src\data\iterative_dmatrix.cc:53: Finished constructing the `IterativeDMatrix`: (16111, 30, 483330).
[11:26:55] INFO: C:\actions-runner\_work\xgboost\xgboost\src\data\ellpack_page.cu:167: Ellpack is dense.
[XGBoost CV (All Features)] Evaluation on Train set
  MAE:  20,312.80 €
  RMSE: 28,667.60 €
  R²:   0.9812
----------------------------------------
[XGBoost CV (All Features)] Evaluation on Test set
  MAE:  63,987.55 €
  RMSE: 96,247.19 €
  R²:   0.7877
----------------------------------------
[DEBUG] Call print_evaluation for XGBoost CV (All Features)
[DEBUG] y_true: 4028, y_pred: 4028

Evaluation – XGBoost CV (All Features)
  MAE:  6

e:\_SoftEng\_BeCode\real-estate-price-predictor\utils\model_evaluator.py:98: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for name, group in df.groupby("price_range"):
e:\_SoftEng\_BeCode\real-estate-price-predictor\utils\model_evaluator.py:98: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for name, group in df.groupby("price_range"):
e:\_SoftEng\_BeCode\real-estate-price-predictor\utils\model_evaluator.py:98: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future

'XGBoost CV (Top RF Features)_20250712_1126.pkl'

In [ ]:

import sys, os

# Add the project root to the Python path
project_root = os.path.abspath("../../..")
sys.path.append(project_root)

import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold

from utils.constants import ML_READY_DATA_FILE, TEST_MODE
from utils.data_loader import DataLoader
from utils.train_test_metrics_logger import TrainTestMetricsLogger
from utils.model_evaluator import ModelEvaluator
from utils.model_saver import ModelSaver

def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

if TEST_MODE:
    print("TEST_MODE is ON – reduced data and iterations.")
else:
    print("TEST_MODE is OFF – full training.")

# === Load and prepare data ===
loader = DataLoader(ML_READY_DATA_FILE)
df = loader.load_data()
X = df.drop(columns=["price"])
y = df["price"]

# === Feature selection ===
selector = VarianceThreshold(threshold=0.01)
selector.fit(X)
X_reduced = X.loc[:, selector.get_support()]

# === Top 30 features via Random Forest ===
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_reduced, y)
top_features = pd.Series(rf.feature_importances_, index=X_reduced.columns).nlargest(30).index.tolist()
X_top = X_reduced[top_features]

# === Split data ===
X_dev_all, X_test_all, y_dev_all, y_test = train_test_split(X_reduced, y, test_size=0.2, random_state=42)
X_train_all, X_val_all, y_train_all, y_val_all = train_test_split(X_dev_all, y_dev_all, test_size=0.2, random_state=42)

X_dev_top, X_test_top, _, _ = train_test_split(X_top, y, test_size=0.2, random_state=42)
X_train_top, X_val_top, y_train_top, y_val_top = train_test_split(X_dev_top, y_dev_all, test_size=0.2, random_state=42)

# === XGBoost Parameters ===
use_gpu = True
params = {
    "objective": "reg:squarederror",
    "random_state": 42,
    "max_depth": 4,
    "min_child_weight": 6,
    "gamma": 0.4,
    "subsample": 0.6,
    "colsample_bytree": 0.6,
    "reg_alpha": 4.0,
    "reg_lambda": 5.0,
    "tree_method": "gpu_hist" if use_gpu else "hist",
    "n_jobs": -1,
    "verbosity": 1
}

n_estimators = 100 if TEST_MODE else 600
early_stopping = 10 if TEST_MODE else 50

# === Train on all features ===
model_all = xgb.XGBRegressor(
    **params,
    n_estimators=n_estimators,
    eval_metric="rmse",
    early_stopping_rounds=early_stopping
)

model_all.fit(
    X_train_all,
    y_train_all,
    eval_set=[(X_val_all, y_val_all)]
)

y_pred_train_all = model_all.predict(X_train_all)
y_pred_test_all = model_all.predict(X_test_all)

# === Train on top RF features ===
model_top = xgb.XGBRegressor(
    **params,
    n_estimators=n_estimators,
    eval_metric="rmse",
    early_stopping_rounds=early_stopping
)

model_top.fit(
    X_train_top,
    y_train_top,
    eval_set=[(X_val_top, y_val_top)]
)

y_pred_train_top = model_top.predict(X_train_top)
y_pred_test_top = model_top.predict(X_test_top)

# === Evaluation ===
price_bins = [0, 250_000, 500_000, 750_000, 1_000_000, float("inf")]

evaluator_all = ModelEvaluator(f"XGBoost CV (All Features) [v6]{' [TEST]' if TEST_MODE else ''}")
metrics_all, _ = evaluator_all.evaluate(y_test, y_pred_test_all, bins=price_bins, dataset_type="Test")
evaluator_all.print_evaluation(y_test, y_pred_test_all, bins=price_bins)

evaluator_top = ModelEvaluator(f"XGBoost CV (Top RF Features) [v6]{' [TEST]' if TEST_MODE else ''}")
metrics_top, _ = evaluator_top.evaluate(y_test, y_pred_test_top, bins=price_bins, dataset_type="Test")
evaluator_top.print_evaluation(y_test, y_pred_test_top, bins=price_bins)

# === Log metrics ===
logger = TrainTestMetricsLogger()

logger.log(
    model_name=f"XGBoost CV (All Features) [v6]{' [TEST]' if TEST_MODE else ''}",
    experiment_name=f"XGBoost FineTuned (All Features){' [TEST]' if TEST_MODE else ''}",
    mae_train=metrics_train_all["mae"],
    rmse_train=metrics_train_all["rmse"],
    r2_train=metrics_train_all["r2"],
    mae_test=metrics_all["mae"],
    rmse_test=metrics_all["rmse"],
    r2_test=metrics_all["r2"],
    data_file=ML_READY_DATA_FILE,
    n_features=X_train_all.shape[1]
)

logger.log(
    model_name=f"XGBoost CV (Top RF Features) [v6]{' [TEST]' if TEST_MODE else ''}",
    experiment_name=f"XGBoost FineTuned (Top RF Features){' [TEST]' if TEST_MODE else ''}",
    mae_train=metrics_train_top["mae"],
    rmse_train=metrics_train_top["rmse"],
    r2_train=metrics_train_top["r2"],
    mae_test=metrics_top["mae"],
    rmse_test=metrics_top["rmse"],
    r2_test=metrics_top["r2"],
    data_file=ML_READY_DATA_FILE,
    n_features=X_train_top.shape[1]
)


# === Step 9: Save models and features ===
saver = ModelSaver()

saver.save_model_and_features(
    model=model_all,
    features=X_train_all.columns.tolist(),
    model_name=f"XGBoost CV (All Features) [v6]{' [TEST]' if TEST_MODE else ''}",
    metrics=metrics_test_all,
    metrics_by_price_range=evaluator_all._compute_metrics_by_price_range(y_test, y_pred_test_all, bins=price_bins)
)

saver.save_model_and_features(
    model=model_top,
    features=X_train_top.columns.tolist(),
    model_name=f"XGBoost CV (Top RF Features) [v6]{' [TEST]' if TEST_MODE else ''}",
    metrics=metrics_test_top,
    metrics_by_price_range=evaluator_top._compute_metrics_by_price_range(y_test, y_pred_test_top, bins=price_bins)
)


# === Display results ===
logger.display_table()


TEST_MODE is OFF – full training.
[0]	validation_0-rmse:183755.28281
[1]	validation_0-rmse:169455.10999
[2]	validation_0-rmse:155558.54114
[3]	validation_0-rmse:145507.87712
[4]	validation_0-rmse:138389.35650
[5]	validation_0-rmse:133416.24968
[6]	validation_0-rmse:130816.55798
[7]	validation_0-rmse:128781.86216
[8]	validation_0-rmse:127169.73801
[9]	validation_0-rmse:126282.86158
[10]	validation_0-rmse:123004.39023
[11]	validation_0-rmse:122350.41856
[12]	validation_0-rmse:120376.11072
[13]	validation_0-rmse:119757.72637
[14]	validation_0-rmse:117893.94152
[15]	validation_0-rmse:117183.56827
[16]	validation_0-rmse:116506.59889
[17]	validation_0-rmse:115746.64992
[18]	validation_0-rmse:115290.60126
[19]	validation_0-rmse:115476.29480
[20]	validation_0-rmse:115219.58396
[21]	validation_0-rmse:114521.26906
[22]	validation_0-rmse:114174.45504
[23]	validation_0-rmse:113613.33058
[24]	validation_0-rmse:113158.37135
[25]	validation_0-rmse:112929.11759
[26]	validation_0-rmse:112536.94981
[27]

e:\_SoftEng\_BeCode\real-estate-price-predictor\.venv\Lib\site-packages\xgboost\callback.py:386: UserWarning: [11:27:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  self.starting_round = model.num_boosted_rounds()


[35]	validation_0-rmse:110258.68988
[36]	validation_0-rmse:109876.21956
[37]	validation_0-rmse:109785.61216
[38]	validation_0-rmse:109614.95812
[39]	validation_0-rmse:109485.33393
[40]	validation_0-rmse:109337.71707
[41]	validation_0-rmse:109231.65934
[42]	validation_0-rmse:109107.36719
[43]	validation_0-rmse:109089.84411
[44]	validation_0-rmse:109057.33692
[45]	validation_0-rmse:108933.72825
[46]	validation_0-rmse:108859.54051
[47]	validation_0-rmse:108479.55799
[48]	validation_0-rmse:108389.92634
[49]	validation_0-rmse:108201.47183
[50]	validation_0-rmse:108050.76798
[51]	validation_0-rmse:107856.02420
[52]	validation_0-rmse:107880.06649
[53]	validation_0-rmse:107820.61433
[54]	validation_0-rmse:107806.07516
[55]	validation_0-rmse:107751.17630
[56]	validation_0-rmse:107701.85837
[57]	validation_0-rmse:107504.35225
[58]	validation_0-rmse:107539.45916
[59]	validation_0-rmse:107333.39435
[60]	validation_0-rmse:107400.08031
[61]	validation_0-rmse:107370.80774
[62]	validation_0-rmse:10729

e:\_SoftEng\_BeCode\real-estate-price-predictor\.venv\Lib\site-packages\xgboost\core.py:2676: UserWarning: [11:27:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  if len(data.shape) != 1 and self.num_features() != data.shape[1]:
e:\_SoftEng\_BeCode\real-estate-price-predictor\.venv\Lib\site-packages\xgboost\callback.py:386: UserWarning: [11:27:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  self.starting_round = model.num_boosted_rounds()


[28]	validation_0-rmse:112508.98693
[29]	validation_0-rmse:112170.13887
[30]	validation_0-rmse:111993.13829
[31]	validation_0-rmse:111781.59779
[32]	validation_0-rmse:111784.36158
[33]	validation_0-rmse:111602.39123
[34]	validation_0-rmse:111294.64163
[35]	validation_0-rmse:111051.91312
[36]	validation_0-rmse:110748.75452
[37]	validation_0-rmse:110522.16944
[38]	validation_0-rmse:110230.93461
[39]	validation_0-rmse:109829.31761
[40]	validation_0-rmse:109723.74283
[41]	validation_0-rmse:109431.29198
[42]	validation_0-rmse:109248.77367
[43]	validation_0-rmse:109056.97233
[44]	validation_0-rmse:108846.84278
[45]	validation_0-rmse:108423.52968
[46]	validation_0-rmse:108314.60648
[47]	validation_0-rmse:108196.30593
[48]	validation_0-rmse:107975.90026
[49]	validation_0-rmse:107551.59526
[50]	validation_0-rmse:107520.62197
[51]	validation_0-rmse:107554.74216
[52]	validation_0-rmse:107491.54417
[53]	validation_0-rmse:107343.17548
[54]	validation_0-rmse:107196.74137
[55]	validation_0-rmse:10717

e:\_SoftEng\_BeCode\real-estate-price-predictor\.venv\Lib\site-packages\xgboost\core.py:2676: UserWarning: [11:27:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  if len(data.shape) != 1 and self.num_features() != data.shape[1]:
e:\_SoftEng\_BeCode\real-estate-price-predictor\utils\model_evaluator.py:98: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for name, group in df.groupby("price_range"):
e:\_SoftEng\_BeCode\real-estate-price-predictor\utils\model_evaluator.py:98: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed

Rank,Best,timestamp,model,mae_train,rmse_train,r2_train,mae_test,rmse_test,r2_test,r2_gap,r2_gap_diagnostic,n_features,"interpretation (r2,mae_gap)",ranking_score
1,✔,2025-07-12 02:26:07,CatBoost + Optuna CV (All Features – Post-Split Evaluation),41.2 k€,56.8 k€,0.926001,61.1 k€,91.4 k€,0.808525,0.117476,Moderate overfitting,72,overfitting,-152492.956535
2,,2025-07-12 02:26:07,CatBoost + Optuna CV(Top RF Features – Post-Split Evaluation),44.6 k€,61.6 k€,0.912976,61.9 k€,91.6 k€,0.807559,0.105417,Moderate overfitting,30,overfitting,-153485.115075
3,,2025-07-12 02:38:07,CatBoost + Optuna CV (Top RF Features – R² gap evaluated),49.7 k€,69.4 k€,0.889556,62.6 k€,92.6 k€,0.803436,0.086120,Moderate overfitting,30,overfitting,-155187.169580
4,,2025-07-12 02:38:07,CatBoost + Optuna CV (All Features – R² gap evaluated),51.7 k€,72.5 k€,0.879752,62.9 k€,93.5 k€,0.799728,0.080024,Moderate overfitting,71,overfitting,-156421.912870
5,,2025-07-12 02:56:11,XGBoost + Optuna CV (All Features) [v6],47.1 k€,69.2 k€,0.890217,62.7 k€,94.2 k€,0.796651,0.093566,Moderate overfitting,72,overfitting,-156912.833580
6,,2025-07-12 02:48:10,XGBoost + Optuna CV (All Features),49.3 k€,72.2 k€,0.880674,63.0 k€,94.8 k€,0.793928,0.086746,Moderate overfitting,72,overfitting,-157860.696947
7,,2025-07-12 02:43:28,XGBoost CV (Top RF Features),22.1 k€,31.4 k€,0.977413,63.7 k€,96.3 k€,0.787667,0.189746,Strong overfitting,30,overfitting,-159934.167991
8,,2025-07-12 11:26:56,XGBoost CV (Top RF Features),22.1 k€,31.4 k€,0.977413,63.7 k€,96.3 k€,0.787667,0.189746,Strong overfitting,30,overfitting,-159934.167991
9,,2025-07-12 02:43:28,XGBoost CV (All Features),20.3 k€,28.7 k€,0.981173,64.0 k€,96.2 k€,0.787700,0.193473,Strong overfitting,72,overfitting,-160233.170278
10,,2025-07-12 11:26:56,XGBoost CV (All Features),20.3 k€,28.7 k€,0.981173,64.0 k€,96.2 k€,0.787700,0.193473,Strong overfitting,72,overfitting,-160233.170278


# 🎯 XGBoost Regression with Optuna Hyperparameter Tuning

This notebook trains two XGBoost regression models on real estate data, with **hyperparameter tuning using Optuna**. It includes all stages from loading the data to model diagnostics.

## Data Preparation

- Load the cleaned ML-ready dataset from a CSV file using `DataLoader`.
- Drop the target variable `price` to separate `X` and `y`.
- Apply `VarianceThreshold` to remove low-variance features (threshold = 0.01).
- Use a `RandomForestRegressor` to rank feature importance.
- Select the **top 30 most important features** for one of the models.


## Hyperparameter Tuning (Optuna)

Define the function `tune_xgboost_with_optuna(...)` that:

- Runs an Optuna optimization loop.
- Evaluates model performance with **5-Fold Cross-Validation**.
- Minimizes the **Root Mean Squared Error (RMSE)**.

### Tuned Hyperparameters:

- `max_depth`
- `learning_rate`
- `n_estimators`
- `subsample`, `colsample_bytree`
- `reg_alpha`, `reg_lambda`
- `min_child_weight`, `gamma`



## Train Final Models

Two models are trained:

- One using **all filtered features**
- One using the **top 30 features**

Each is trained using the **best parameters** found by Optuna.

---

## Evaluation

Models are evaluated using:

- `MAE`: Mean Absolute Error  
- `RMSE`: Root Mean Squared Error  
- `R<sup>2</sup>`: Coefficient of determination  

Results are logged with `ExperimentTracker`.



## Diagnostics

- Summary tables displayed with `ModelEvaluator`
- Residuals & diagnostic plots from `ModelVisualizer`
- Optionally, **SHAP values** can be plotted to understand feature importance



## Test Mode (Optional)

When `TEST_MODE = True`, the pipeline uses:

- A smaller dataset  
- Fewer Optuna trials (`n_trials = 3`)  

To speed up execution and debugging.


In [ ]:
import sys, os
# Add the project root to the Python path
project_root = os.path.abspath("../../..")
sys.path.append(project_root)

import optuna
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold

from utils.constants import ML_READY_DATA_FILE, TEST_MODE
from utils.data_loader import DataLoader
from utils.train_test_metrics_logger import TrainTestMetricsLogger
from utils.model_saver import ModelSaver

def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

if TEST_MODE:
    print("TEST_MODE is ON – reduced data and iterations.")
else:
    print("TEST_MODE is OFF – full training.")

# === Load dataset ===
loader = DataLoader(ML_READY_DATA_FILE)
df = loader.load_data()

X = df.drop(columns=["price"])
y = df["price"]

# === Feature Selection ===
selector = VarianceThreshold(threshold=0.01)
selector.fit(X)
X_reduced = X.loc[:, selector.get_support()]

# === Top 30 features with Random Forest ===
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_reduced, y)
top_features = pd.Series(rf_model.feature_importances_, index=X_reduced.columns).nlargest(30).index.tolist()
X_top = X_reduced[top_features]

# === Split datasets ===
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(X_reduced, y, test_size=0.2, random_state=42)
X_train_top, X_test_top, y_train_top, y_test_top = train_test_split(X_top, y, test_size=0.2, random_state=42)

# === GPU usage ===
use_gpu = True
device = "cuda" if use_gpu else "cpu"
tree_method = "gpu_hist" if use_gpu else "hist"
random_state = 42
n_trials = 3 if TEST_MODE else 50
early_stopping_rounds = 10 if TEST_MODE else 50



# === Optuna Tuning ===
def tune_xgboost_with_optuna(X_data, y_data, n_trials):
    def objective(trial):
        params = {
            "max_depth": trial.suggest_int("max_depth", 2, 6),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "subsample": trial.suggest_float("subsample", 0.6, 0.9),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.1, 5.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 5.0),
            "min_child_weight": trial.suggest_float("min_child_weight", 5, 20),
            "gamma": trial.suggest_float("gamma", 0, 5),
            "tree_method": tree_method,
            "device": device,
            "random_state": random_state,
            "objective": "reg:squarederror",
            "n_jobs": -1,
            "verbosity": 0,
        }
        model = xgb.XGBRegressor(**params)
        cv = KFold(n_splits=5, shuffle=True, random_state=random_state)
        scores = -cross_val_score(model, X_data, y_data, scoring="neg_root_mean_squared_error", cv=cv)
        return scores.mean()

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

# === Train & Evaluate ===
def train_and_evaluate(X_train_full, y_train_full, X_test, y_test, model_name, experiment_name):
    X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

    best_params = tune_xgboost_with_optuna(X_train, y_train, n_trials)
    
    model = xgb.XGBRegressor(
        **best_params,
        tree_method=tree_method,
        device=device,
        random_state=random_state,
        objective="reg:squarederror",
        n_jobs=-1,
        verbosity=0,
        eval_metric="rmse",
        early_stopping_rounds=early_stopping_rounds
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    y_pred_train = model.predict(X_train_full)
    y_pred_test = model.predict(X_test)

    mae_train = mean_absolute_error(y_train_full, y_pred_train)
    rmse_train = root_mean_squared_error(y_train_full, y_pred_train)
    r2_train = r2_score(y_train_full, y_pred_train)

    mae_test = mean_absolute_error(y_test, y_pred_test)
    rmse_test = root_mean_squared_error(y_test, y_pred_test)
    r2_test = r2_score(y_test, y_pred_test)

    logger.log(
        model_name=model_name,
        experiment_name=experiment_name,
        mae_train=mae_train,
        rmse_train=rmse_train,
        r2_train=r2_train,
        mae_test=mae_test,
        rmse_test=rmse_test,
        r2_test=r2_test,
        data_file=ML_READY_DATA_FILE,
        n_features=X_train_full.shape[1]
    )

    print(f"{model_name} trained and logged.")
    print(f"Train R²: {r2_train:.4f}, Test R²: {r2_test:.4f}")

# === Logger ===
logger = TrainTestMetricsLogger()

# === Run All Features ===
train_and_evaluate(
    X_train_all, y_train_all, X_test_all, y_test_all,
    model_name=f"XGBoost + Optuna CV (All Features){' [TEST]' if TEST_MODE else ''}",
    experiment_name=f"XGBoost FineTuned (All Features){' [TEST]' if TEST_MODE else ''}"
)

# === Run Top 30 Features ===
train_and_evaluate(
    X_train_top, y_train_top, X_test_top, y_test_top,
    model_name=f"XGBoost + Optuna CV (Top 30 Features){' [TEST]' if TEST_MODE else ''}",
    experiment_name=f"XGBoost FineTuned (Top 30 Features){' [TEST]' if TEST_MODE else ''}"
)

# === Display Logs ===
logger.display_table()


TEST_MODE is OFF – full training.


[I 2025-07-12 11:27:17,263] A new study created in memory with name: no-name-0912f92d-3b3d-453b-9050-36f915d17d22
[I 2025-07-12 11:27:18,515] Trial 0 finished with value: 113895.97550527714 and parameters: {'max_depth': 2, 'learning_rate': 0.13445122767382223, 'n_estimators': 107, 'subsample': 0.7119304145089252, 'colsample_bytree': 0.8382780492495678, 'reg_alpha': 1.8944413616356628, 'reg_lambda': 1.4795900439361263, 'min_child_weight': 7.3034367940139076, 'gamma': 3.2063980002962396}. Best is trial 0 with value: 113895.97550527714.
[I 2025-07-12 11:27:25,099] Trial 1 finished with value: 99279.82163466219 and parameters: {'max_depth': 6, 'learning_rate': 0.1883834111569362, 'n_estimators': 412, 'subsample': 0.8331197479898835, 'colsample_bytree': 0.820290833550271, 'reg_alpha': 0.16251020590547405, 'reg_lambda': 3.4858224651584613, 'min_child_weight': 19.59535964451228, 'gamma': 3.8950448259277577}. Best is trial 1 with value: 99279.82163466219.
[I 2025-07-12 11:27:28,137] Trial 2 fi

XGBoost + Optuna CV (All Features) trained and logged.
Train R²: 0.8762, Test R²: 0.7957


[I 2025-07-12 11:31:10,666] Trial 0 finished with value: 101117.35962054075 and parameters: {'max_depth': 5, 'learning_rate': 0.038820958416807286, 'n_estimators': 286, 'subsample': 0.8197396062117002, 'colsample_bytree': 0.6139727437465076, 'reg_alpha': 1.1455886112017837, 'reg_lambda': 3.5333166456251126, 'min_child_weight': 15.75391793728252, 'gamma': 3.6493232406739695}. Best is trial 0 with value: 101117.35962054075.
[I 2025-07-12 11:31:11,977] Trial 1 finished with value: 107627.39564175686 and parameters: {'max_depth': 2, 'learning_rate': 0.1441623973176674, 'n_estimators': 204, 'subsample': 0.651856120000821, 'colsample_bytree': 0.643822086432025, 'reg_alpha': 3.6617825046451706, 'reg_lambda': 2.419451371391697, 'min_child_weight': 15.884283454018831, 'gamma': 1.0792915417140754}. Best is trial 0 with value: 101117.35962054075.
[I 2025-07-12 11:31:13,224] Trial 2 finished with value: 103064.9050720897 and parameters: {'max_depth': 3, 'learning_rate': 0.18211771096497026, 'n_est

XGBoost + Optuna CV (Top 30 Features) trained and logged.
Train R²: 0.8604, Test R²: 0.7885


Rank,Best,timestamp,model,mae_train,rmse_train,r2_train,mae_test,rmse_test,r2_test,r2_gap,r2_gap_diagnostic,n_features,"interpretation (r2,mae_gap)",ranking_score
1,✔,2025-07-12 02:26:07,CatBoost + Optuna CV (All Features – Post-Split Evaluation),41.2 k€,56.8 k€,0.926001,61.1 k€,91.4 k€,0.808525,0.117476,Moderate overfitting,72,overfitting,-152492.956535
2,,2025-07-12 02:26:07,CatBoost + Optuna CV(Top RF Features – Post-Split Evaluation),44.6 k€,61.6 k€,0.912976,61.9 k€,91.6 k€,0.807559,0.105417,Moderate overfitting,30,overfitting,-153485.115075
3,,2025-07-12 02:38:07,CatBoost + Optuna CV (Top RF Features – R² gap evaluated),49.7 k€,69.4 k€,0.889556,62.6 k€,92.6 k€,0.803436,0.086120,Moderate overfitting,30,overfitting,-155187.169580
4,,2025-07-12 02:38:07,CatBoost + Optuna CV (All Features – R² gap evaluated),51.7 k€,72.5 k€,0.879752,62.9 k€,93.5 k€,0.799728,0.080024,Moderate overfitting,71,overfitting,-156421.912870
5,,2025-07-12 02:56:11,XGBoost + Optuna CV (All Features) [v6],47.1 k€,69.2 k€,0.890217,62.7 k€,94.2 k€,0.796651,0.093566,Moderate overfitting,72,overfitting,-156912.833580
6,,2025-07-12 11:31:07,XGBoost + Optuna CV (All Features),50.2 k€,73.5 k€,0.876212,62.8 k€,94.4 k€,0.795743,0.080469,Moderate overfitting,72,overfitting,-157164.931713
7,,2025-07-12 02:48:10,XGBoost + Optuna CV (All Features),49.3 k€,72.2 k€,0.880674,63.0 k€,94.8 k€,0.793928,0.086746,Moderate overfitting,72,overfitting,-157860.696947
8,,2025-07-12 02:43:28,XGBoost CV (Top RF Features),22.1 k€,31.4 k€,0.977413,63.7 k€,96.3 k€,0.787667,0.189746,Strong overfitting,30,overfitting,-159934.167991
9,,2025-07-12 11:26:56,XGBoost CV (Top RF Features),22.1 k€,31.4 k€,0.977413,63.7 k€,96.3 k€,0.787667,0.189746,Strong overfitting,30,overfitting,-159934.167991
10,,2025-07-12 02:43:28,XGBoost CV (All Features),20.3 k€,28.7 k€,0.981173,64.0 k€,96.2 k€,0.787700,0.193473,Strong overfitting,72,overfitting,-160233.170278


In [ ]:
import sys, os

# Add the project root to the Python path
project_root = os.path.abspath("../../..")
sys.path.append(project_root)

import optuna
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold

from utils.constants import ML_READY_DATA_FILE, TEST_MODE
from utils.data_loader import DataLoader
from utils.train_test_metrics_logger import TrainTestMetricsLogger
from utils.model_saver import ModelSaver
from utils.model_evaluator import ModelEvaluator


# === Helpers ===
def root_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


# === Optuna Tuning ===
def tune_xgboost_with_optuna(X, y, n_trials):
    def objective(trial):
        params = {
            "max_depth": trial.suggest_int("max_depth", 2, 6),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "subsample": trial.suggest_float("subsample", 0.6, 0.9),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.1, 5.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 5.0),
            "min_child_weight": trial.suggest_float("min_child_weight", 5, 20),
            "gamma": trial.suggest_float("gamma", 0, 5),
            "tree_method": tree_method,
            "device": device,
            "random_state": random_state,
            "objective": "reg:squarederror",
            "n_jobs": -1,
            "verbosity": 0,
        }
        model = xgb.XGBRegressor(**params)
        cv = KFold(n_splits=5, shuffle=True, random_state=random_state)
        score = -cross_val_score(model, X, y, scoring="neg_root_mean_squared_error", cv=cv).mean()
        return score

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)
    return study.best_params


# === Config ===
print("TEST_MODE =", TEST_MODE)
use_gpu = True
device = "cuda" if use_gpu else "cpu"
tree_method = "gpu_hist" if use_gpu else "hist"
random_state = 42
n_trials = 3 if TEST_MODE else 50
early_stopping_rounds = 10 if TEST_MODE else 50

# === Load & Prepare Data ===
loader = DataLoader(ML_READY_DATA_FILE)
df = loader.load_data()
X = df.drop(columns=["price"])
y = df["price"]

# Variance Threshold
selector = VarianceThreshold(threshold=0.01)
selector.fit(X)
X_reduced = X.loc[:, selector.get_support()]

# Top 30 Features (Random Forest)
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_reduced, y)
top_30 = pd.Series(rf.feature_importances_, index=X_reduced.columns).nlargest(30).index.tolist()
X_top = X_reduced[top_30]

# Train-Test Split
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(X_reduced, y, test_size=0.2, random_state=42)
X_train_top, X_test_top, y_train_top, y_test_top = train_test_split(X_top, y, test_size=0.2, random_state=42)

# === Logger ===
logger = TrainTestMetricsLogger()


# === Train and Evaluate Pipeline ===
def train_and_evaluate(X_train_full, y_train_full, X_test, y_test, model_label, experiment_label):
    X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

    best_params = tune_xgboost_with_optuna(X_train, y_train, n_trials)
    model = xgb.XGBRegressor(
        **best_params,
        tree_method=tree_method,
        device=device,
        random_state=random_state,
        objective="reg:squarederror",
        n_jobs=-1,
        verbosity=0,
        eval_metric="rmse",
        early_stopping_rounds=early_stopping_rounds
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    y_pred_train = model.predict(X_train_full)
    y_pred_test = model.predict(X_test)

    # Metrics globales
    metrics_train = {
        "mae": mean_absolute_error(y_train_full, y_pred_train),
        "rmse": root_rmse(y_train_full, y_pred_train),
        "r2": r2_score(y_train_full, y_pred_train)
    }
    metrics_test = {
        "mae": mean_absolute_error(y_test, y_pred_test),
        "rmse": root_rmse(y_test, y_pred_test),
        "r2": r2_score(y_test, y_pred_test)
    }

    # Segmented evaluation via ModelEvaluator
    evaluator = ModelEvaluator(model_label)
    _, metrics_by_price_range = evaluator.evaluate(y_test, y_pred_test, bins=[0, 150000, 250000, 350000, 500000, 750000, 1000000, float("inf")])

    # Logging
    logger.log(
        model_name=f"{model_label} [v6]{' [TEST]' if TEST_MODE else ''}",
        experiment_name=f"{experiment_label}{' [TEST]' if TEST_MODE else ''}",
        mae_train=metrics_train["mae"],
        rmse_train=metrics_train["rmse"],
        r2_train=metrics_train["r2"],
        mae_test=metrics_test["mae"],
        rmse_test=metrics_test["rmse"],
        r2_test=metrics_test["r2"],
        data_file=ML_READY_DATA_FILE,
        n_features=X_train_full.shape[1]
    )

    # Save model
    saver = ModelSaver()
    saver.save_model_and_features(
        model=model,
        features=X_train_full.columns.tolist(),
        model_name=f"{model_label} [v6]{' [TEST]' if TEST_MODE else ''}",
        metrics=metrics_test,
        metrics_by_price_range=metrics_by_price_range  
    )

    print(f"{model_label} trained and logged. R² Test: {metrics_test['r2']:.4f}")
    return model


# === Run Training for Both Sets ===
train_and_evaluate(
    X_train_all, y_train_all, X_test_all, y_test_all,
    model_label="XGBoost + Optuna CV (All Features)",
    experiment_label="XGBoost FineTuned (All Features)"
)

train_and_evaluate(
    X_train_top, y_train_top, X_test_top, y_test_top,
    model_label="XGBoost + Optuna CV (Top 30 Features)",
    experiment_label="XGBoost FineTuned (Top 30 Features)"
)

# === Show Summary Table ===
logger.display_table()


TEST_MODE = False


[I 2025-07-12 11:34:58,519] A new study created in memory with name: no-name-cd97b13a-dee9-45db-9620-2be621ec5aab
[I 2025-07-12 11:35:00,421] Trial 0 finished with value: 98283.55805501185 and parameters: {'max_depth': 5, 'learning_rate': 0.16796890887359125, 'n_estimators': 137, 'subsample': 0.8233033647991093, 'colsample_bytree': 0.6281861231633105, 'reg_alpha': 4.902470895864407, 'reg_lambda': 4.849053335047096, 'min_child_weight': 16.112440203220636, 'gamma': 3.3549534139312738}. Best is trial 0 with value: 98283.55805501185.
[I 2025-07-12 11:35:02,145] Trial 1 finished with value: 106656.00317177191 and parameters: {'max_depth': 2, 'learning_rate': 0.11986565957604207, 'n_estimators': 251, 'subsample': 0.7454319469551568, 'colsample_bytree': 0.8032897311214087, 'reg_alpha': 4.950574249631151, 'reg_lambda': 2.2256844548138246, 'min_child_weight': 16.511199288132374, 'gamma': 1.583950476344134}. Best is trial 0 with value: 98283.55805501185.
[I 2025-07-12 11:35:04,531] Trial 2 finis

[✔] Model saved: XGBoost + Optuna CV (All Features) [v6]_20250712_1138.pkl
[✔] Features saved: XGBoost + Optuna CV (All Features) [v6]_20250712_1138.json
[✔] Metrics saved: XGBoost + Optuna CV (All Features) [v6]_20250712_1138_metrics.json
[✔] Price range metrics saved: XGBoost + Optuna CV (All Features) [v6]_20250712_1138_metrics_by_price_range.json
XGBoost + Optuna CV (All Features) trained and logged. R² Test: 0.7975


[I 2025-07-12 11:38:53,615] Trial 0 finished with value: 102654.33183546088 and parameters: {'max_depth': 2, 'learning_rate': 0.18006380588037618, 'n_estimators': 406, 'subsample': 0.654514439897528, 'colsample_bytree': 0.8078245231312245, 'reg_alpha': 1.6641335840146612, 'reg_lambda': 0.40335941792834673, 'min_child_weight': 8.803789010679862, 'gamma': 2.7294586351935326}. Best is trial 0 with value: 102654.33183546088.
[I 2025-07-12 11:38:54,957] Trial 1 finished with value: 112171.85372117495 and parameters: {'max_depth': 3, 'learning_rate': 0.05347648826277473, 'n_estimators': 151, 'subsample': 0.7322722201324886, 'colsample_bytree': 0.6549437050335336, 'reg_alpha': 0.38070289415576286, 'reg_lambda': 4.095259154495855, 'min_child_weight': 11.407764268045739, 'gamma': 3.752073545565988}. Best is trial 0 with value: 102654.33183546088.
[I 2025-07-12 11:39:00,834] Trial 2 finished with value: 98082.30708935182 and parameters: {'max_depth': 6, 'learning_rate': 0.05171950133492538, 'n_e

[✔] Model saved: XGBoost + Optuna CV (Top 30 Features) [v6]_20250712_1142.pkl
[✔] Features saved: XGBoost + Optuna CV (Top 30 Features) [v6]_20250712_1142.json
[✔] Metrics saved: XGBoost + Optuna CV (Top 30 Features) [v6]_20250712_1142_metrics.json
[✔] Price range metrics saved: XGBoost + Optuna CV (Top 30 Features) [v6]_20250712_1142_metrics_by_price_range.json
XGBoost + Optuna CV (Top 30 Features) trained and logged. R² Test: 0.7885


e:\_SoftEng\_BeCode\real-estate-price-predictor\utils\model_evaluator.py:98: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for name, group in df.groupby("price_range"):


Rank,Best,timestamp,model,mae_train,rmse_train,r2_train,mae_test,rmse_test,r2_test,r2_gap,r2_gap_diagnostic,n_features,"interpretation (r2,mae_gap)",ranking_score
1,✔,2025-07-12 02:26:07,CatBoost + Optuna CV (All Features – Post-Split Evaluation),41.2 k€,56.8 k€,0.926001,61.1 k€,91.4 k€,0.808525,0.117476,Moderate overfitting,72,overfitting,-152492.956535
2,,2025-07-12 02:26:07,CatBoost + Optuna CV(Top RF Features – Post-Split Evaluation),44.6 k€,61.6 k€,0.912976,61.9 k€,91.6 k€,0.807559,0.105417,Moderate overfitting,30,overfitting,-153485.115075
3,,2025-07-12 02:38:07,CatBoost + Optuna CV (Top RF Features – R² gap evaluated),49.7 k€,69.4 k€,0.889556,62.6 k€,92.6 k€,0.803436,0.086120,Moderate overfitting,30,overfitting,-155187.169580
4,,2025-07-12 02:38:07,CatBoost + Optuna CV (All Features – R² gap evaluated),51.7 k€,72.5 k€,0.879752,62.9 k€,93.5 k€,0.799728,0.080024,Moderate overfitting,71,overfitting,-156421.912870
5,,2025-07-12 02:56:11,XGBoost + Optuna CV (All Features) [v6],47.1 k€,69.2 k€,0.890217,62.7 k€,94.2 k€,0.796651,0.093566,Moderate overfitting,72,overfitting,-156912.833580
6,,2025-07-12 11:38:51,XGBoost + Optuna CV (All Features) [v6],48.8 k€,70.9 k€,0.884973,63.0 k€,94.0 k€,0.797469,0.087504,Moderate overfitting,72,overfitting,-156956.458866
7,,2025-07-12 11:31:07,XGBoost + Optuna CV (All Features),50.2 k€,73.5 k€,0.876212,62.8 k€,94.4 k€,0.795743,0.080469,Moderate overfitting,72,overfitting,-157164.931713
8,,2025-07-12 02:48:10,XGBoost + Optuna CV (All Features),49.3 k€,72.2 k€,0.880674,63.0 k€,94.8 k€,0.793928,0.086746,Moderate overfitting,72,overfitting,-157860.696947
9,,2025-07-12 02:43:28,XGBoost CV (Top RF Features),22.1 k€,31.4 k€,0.977413,63.7 k€,96.3 k€,0.787667,0.189746,Strong overfitting,30,overfitting,-159934.167991
10,,2025-07-12 11:26:56,XGBoost CV (Top RF Features),22.1 k€,31.4 k€,0.977413,63.7 k€,96.3 k€,0.787667,0.189746,Strong overfitting,30,overfitting,-159934.167991


# Saving XGBoost + Optuna Hyperparameter Tuning Models (`.pkl`) After Training

After training XGBoost models with Optuna tuning, it's essential to persist the trained models using `.pkl` files. The script below ensures each model is saved with a unique, timestamped filename and organized in the correct directory.


##  What the Script Does

1. **Appends the project root** to the Python path (to allow relative imports).
2. **Generates a timestamped filename**, including an optional `_TEST` suffix if `TEST_MODE` is enabled.
3. **Ensures the target directory exists**, and creates it if necessary.
4. **Saves both trained models** using `joblib.dump()`:
   - One trained with **all features**.
   - One trained with the **top 30 features** (e.g., selected via Random Forest).

In [ ]:
import sys, os

# Add the project root to the Python path
project_root = os.path.abspath("../../..")
sys.path.append(project_root)

import joblib
from datetime import datetime
from utils.constants import TEST_MODE, MODELS_DIR

# Create timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# Add suffix if in TEST mode
suffix = "_TEST" if TEST_MODE else ""

# Define subdirectory for .pkl files
PKL_DIR = os.path.join(MODELS_DIR, "pkl")
os.makedirs(PKL_DIR, exist_ok=True)

# Build filenames
filename_all = f"xgboost_optuna_all_{timestamp}{suffix}.pkl"
filename_top = f"xgboost_optuna_top30_{timestamp}{suffix}.pkl"

# Save models
joblib.dump(model_all, os.path.join(PKL_DIR, filename_all))
joblib.dump(model_top, os.path.join(PKL_DIR, filename_top))

print(f"[✔] Models saved to '{PKL_DIR}' as:\n - {filename_all}\n - {filename_top}")


[✔] Models saved to 'e:\_SoftEng\_BeCode\real-estate-price-predictor\models\pkl' as:
 - xgboost_optuna_all_20250712_1142.pkl
 - xgboost_optuna_top30_20250712_1142.pkl


# Saving Feature Lists Used by Each Model (`.json`)

After training and saving your machine learning models (e.g., XGBoost or CatBoost), it's critical to also save the **list of features** used during training. This ensures **inference compatibility** and prevents mismatches between the model and the input data.


## What the Script Does

1. **Creates the directory** for storing feature metadata:
   - Located in: `models/features/`

2. **Saves two JSON files**:
   - One listing the full set of features used in the **all-features model**.
   - One listing the selected **top 30 features** (e.g., based on feature importance).

3. **Uses the same base name as the corresponding `.pkl` model**, replacing the extension:
   - Example: `xgboost_optuna_all_20250629_1430.pkl` → `xgboost_optuna_all_20250629_1430.json`





In [14]:
import json

# Define subdirectory for features
FEATURES_DIR = os.path.join(MODELS_DIR, "features")
os.makedirs(FEATURES_DIR, exist_ok=True)

# Save features used for each model
with open(os.path.join(FEATURES_DIR, filename_all.replace(".pkl", ".json")), "w") as f:
    json.dump(list(X_reduced.columns), f, indent=2)

with open(os.path.join(FEATURES_DIR, filename_top.replace(".pkl", ".json")), "w") as f:
    json.dump(top_features, f, indent=2)

print(f"[✔] Associated feature files saved to '{FEATURES_DIR}'")


[✔] Associated feature files saved to 'e:\_SoftEng\_BeCode\real-estate-price-predictor\models\features'
